# DF-RISE reproduction — Kaggle notebook

**Setup**: Accel = GPU T4 x2. Clone repo first, install deps, run experiments.

This notebook runs the **DF-RISE** tool (paper §3.1). Extend it with DF-CAM, exponential sampling, and the feature-importance variant once this works.

In [ ]:
# 0) Get code + deps
!git clone https://github.com/<YOUR_USER>/dfrise-project.git /kaggle/working/dfrise-project
%cd /kaggle/working/dfrise-project
!pip install -q diffusers transformers accelerate torchvision scikit-image numpy matplotlib pyyaml
!pip install -q piqa torch_fidelity   # SSIM + FID for later; torch_fidelity optional

In [ ]:
import sys; sys.path.insert(0, "/kaggle/working/dfrise-project")
import torch
from src.pipeline import load_stable_diffusion_components, encode_prompt
from src.df_rise import df_rise_step
from src.visualization import plot_step_series, heatmap_overlay
import yaml

cfg = yaml.safe_load(open("configs/default.yaml"))
components = load_stable_diffusion_components(cfg)
unet, vae = components["unet"], components["vae"]
scheduler = components["scheduler"]
device = "cuda"

In [ ]:
# 1) Generate a base image (record which steps we want saliency for)
from src.pipeline import denoise_with_hooks
import numpy as np

PROMPT = "an astronaut riding a horse on mars"
img, info = denoise_with_hooks(components, PROMPT, seed=cfg["model"]["seed"],
                               guidance_scale=cfg["model"]["guidance_scale"])
# show base image (save to outputs/)
import matplotlib.pyplot as plt
plt.imsave("outputs/base_image.png", img)
plt.imshow(img); plt.axis('off'); plt.show()

In [ ]:
# 2) DF-RISE: compute per-step saliency maps
# N_masks per step; run on a subset of steps (e.g. every 5th) to save compute
emb = encode_prompt(components, PROMPT, device)
timesteps = scheduler.timesteps
steps_to_analyze = [i for i in range(0, len(timesteps), 5)]

saliency_maps = []
analyzed_ts = []

def hook(noise_pred, current, t, t_idx, prev):
    if t_idx in steps_to_analyze:
        S = df_rise_step(unet, vae, emb, current, int(t),
                         n_masks=cfg["dfrise"]["n_masks"],
                         guidance_scale=cfg["model"]["guidance_scale"])
        saliency_maps.append(S.cpu().numpy())
        analyzed_ts.append(int(t))

denoise_with_hooks(components, PROMPT,
                   seed=cfg["model"]["seed"],
                   guidance_scale=cfg["model"]["guidance_scale"],
                   step_hook=hook)

fig = plot_step_series(saliency_maps, analyzed_ts, save_path="outputs/dfrise_steps.png")
plt.show()

In [ ]:
# 3) Overlay + save
import numpy as np
from src.visualization import heatmap_overlay
over = heatmap_overlay(img, saliency_maps[-1])
plt.imsave("outputs/dfrise_overlay.png", over)
plt.imshow(over); plt.axis('off'); plt.show()